# FinanceBench RAG Pipeline Notebook

This notebook runs the full FinanceBench pipeline end-to-end using the modular package in `src/financebench_rag`.

Stages covered:
1. Environment and configuration setup
2. Dataset loading and filtering
3. PDF link repair and local map validation
4. Naive baseline generation
5. PDF loading with page-level metadata
6. Chunking and FAISS build/persist
7. Retrieval sanity checks
8. `answer_with_rag(query, k)`
9. Side-by-side comparison
10. Correctness evaluation (LLM-as-judge)
11. Faithfulness evaluation (Ragas, first 20)
12. Retrieval Page Hit@k
13. Artifact export

## 1) Environment and Configuration Setup

Install dependencies first from `requirements.txt`, then set `.env` values based on `.env.example`.

This cell imports package modules and prepares config/paths.

In [ ]:
%pip install datasets langchain langchain-community langchain-core
%pip install faiss-cpu sentence-transformers pypdf requests cryptography
%pip install ragas

Note: you may need to restart the kernel to use updated packages.
  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.11.0-cp312-cp312-win_amd64.whl.metadata (29 kB)
  Using cached huggingface_hub-1.10.2-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached sentence_transformers-5.4.1-py3-none-any.whl (571 kB)
Using cached torch-2.11.0-cp312-cp312-win_amd64.whl (114.6 MB)
Using cached transformers-5.5.4-py3-none-any.whl (10.2 MB)
Using cached huggingface_hub-1.10.2-py3-none-any.whl (642 kB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.33.1
    Uninstalling huggingface_hub-0.33.1:
      Successfully uninstalled huggingface_hub-0.33.1
Note: you may need to restart the kernel to use updated packages.

In [2]:
%pip install -U langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import json
import pandas as pd

from src.financebench_rag.config import load_config
from src.financebench_rag.dataset import (
    prepare_stage1_dataset,
    build_doc_metadata_table,
)
from src.financebench_rag.io_utils import (
    save_dataframe_csv,
    save_dataframe_json,
    save_json,
)
from src.financebench_rag.naive_generation import (
    sample_stage2_questions,
    run_naive_generation,
)
from src.financebench_rag.vectorstore import (
    download_required_pdfs,
    load_pdf_pages_with_metadata,
    chunk_documents,
    build_or_load_vectorstore,
)
from src.financebench_rag.retrieval_checks import run_retrieval_sanity_checks
from src.financebench_rag.rag_pipeline import RAGPipeline
from src.financebench_rag.comparison import build_side_by_side_table
from src.financebench_rag.evaluation import (
    compute_correctness_judgements,
    compute_faithfulness_first_20,
    compute_page_hit_rate,
)

config = load_config()
config.results_dir.mkdir(parents=True, exist_ok=True)
config.pdf_dir.mkdir(parents=True, exist_ok=True)
config.vectorstore_dir.mkdir(parents=True, exist_ok=True)

print("Loaded config")
print(config)

Loaded config
PipelineConfig(dataset_id='PatronusAI/financebench', dataset_split='train', api_base_url='https://api.tokenfactory.nebius.com/v1', api_key='v1.CmQKHHN0YXRpY2tleS1lMDBkZW4xNGJ2aGs1YWowZ2ESIXNlcnZpY2VhY2NvdW50LWUwMHg4ZGVyazh0dHJ5cTFlNDIMCOydpM4GEPPH3eoCOgwI66C8mQcQwOPnmQJAAloDZTAw.AAAAAAAAAAEkdoZogNdkUIy462oJv3-9F7IqWb2jOKx_1MgpIoV8BcE0g41S1owg6ZSpgi9W28qsNb8GO3_ILQYFAW2-DSUF', generation_model='meta-llama/Llama-3.3-70B-Instruct', judge_model='deepseek-ai/DeepSeek-V3.2', ragas_model='deepseek-ai/DeepSeek-V3.2', embedding_model='BAAI/bge-small-en-v1.5', chunk_size=1000, chunk_overlap=150, retrieval_default_k=4, retrieval_hit_k_values=[1, 3, 5], data_dir=WindowsPath('data'), pdf_dir=WindowsPath('data/pdfs'), vectorstore_dir=WindowsPath('vectorstore'), results_dir=WindowsPath('results'))


## 2) Load and Filter FinanceBench Dataset

Load from HuggingFace, keep only domain-relevant and novel-generated, sort by financebench_id, and normalize evidence page numbers.

In [2]:
filtered_df, doc_mapping_df = prepare_stage1_dataset(
    dataset_id=config.dataset_id,
    split=config.dataset_split,
    output_dir=config.results_dir,
)

save_dataframe_csv(filtered_df, config.results_dir / "stage1_filtered.csv")
save_dataframe_json(filtered_df, config.results_dir / "stage1_filtered.json")

print("Filtered rows:", len(filtered_df))
filtered_df[["financebench_id", "question_type", "doc_name"]].head()

Filtered rows: 100


,financebench_id,question_type,doc_name
0,financebench_id_00005,domain-relevant,CORNING_2022_10K
1,financebench_id_00070,domain-relevant,AMERICANWATERWORKS_2022_10K
2,financebench_id_00080,domain-relevant,PAYPAL_2022_10K
3,financebench_id_00206,domain-relevant,JPMORGAN_2022_10K
4,financebench_id_00215,domain-relevant,VERIZON_2022_10K


## 3) Repair PDF Links and Build Local Document Map

Repair dead links using FinanceBench PDF repository path conventions and save mapping artifacts.

## 4) Run Naive Generation Baseline (No Retrieval)

Select first 5 rows of each target question type and run no-context generation.

In [4]:
save_dataframe_csv(doc_mapping_df, config.results_dir / "stage1_doc_mapping.csv")
save_dataframe_json(doc_mapping_df, config.results_dir / "stage1_doc_mapping.json")

stage2_questions = sample_stage2_questions(filtered_df, n_per_type=5)
naive_df = run_naive_generation(stage2_questions, config)

save_dataframe_csv(naive_df, config.results_dir / "stage2_naive.csv")
save_dataframe_json(naive_df, config.results_dir / "stage2_naive.json")

print("Stage 2 sample size:", len(stage2_questions))
naive_df.head()

Stage 2 sample size: 10


,financebench_id,question_type,question,ground_truth,naive_answer
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,"Based on Corning's FY2022 data, I do not know ..."
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",I do not know the working capital of American ...
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"Based on PayPal's FY2022 data, I do not have t..."
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...",Gross margins are not a relevant metric for JP...
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,"Yes, Verizon is a capital-intensive business. ..."


## 5) Load PDFs and Attach Page-Level Metadata

Load only dataset-referenced PDFs and attach `doc_name`, `company`, `doc_period`, `page_number`.

## 6) Chunk Documents and Build/Persist FAISS Vector Store

Chunk with size 1000 and overlap 150, build FAISS using BAAI/bge-small-en-v1.5, and persist for reuse.

## 7) Run Retrieval Sanity Checks

Validate top-k retrieval document/page alignment on 2-3 examples.

In [5]:
doc_table = build_doc_metadata_table(filtered_df)
local_pdfs = download_required_pdfs(doc_table["doc_name"].tolist(), config.pdf_dir)

pages = load_pdf_pages_with_metadata(doc_table, config.pdf_dir)
chunks = chunk_documents(
    pages,
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap,
)
vectorstore = build_or_load_vectorstore(chunks, config)

sanity_df = run_retrieval_sanity_checks(filtered_df.head(3), vectorstore, k=config.retrieval_default_k)
save_dataframe_csv(sanity_df, config.results_dir / "stage3_sanity_checks.csv")
save_dataframe_json(sanity_df, config.results_dir / "stage3_sanity_checks.json")

print("Local PDFs available:", len(local_pdfs))
print("Pages loaded:", len(pages))
print("Chunks created:", len(chunks))
sanity_df.head()

DependencyError: cryptography>=3.1 is required for AES algorithm

## 8) Implement answer_with_rag(query, k)

Instantiate `RAGPipeline` and call `answer_with_rag` with retrieved chunk metadata.

## 9) Generate Side-by-Side Comparison (Ground Truth vs Naive vs RAG)

Run the same 10 baseline questions through RAG and compare outputs.

In [ ]:
rag = RAGPipeline(config=config, vectorstore=vectorstore)

example_rag = rag.answer_with_rag(
    query=str(stage2_questions.iloc[0]["question"]),
    k=config.retrieval_default_k,
)
print("Example RAG answer:")
print(example_rag["answer"])
print("Retrieved chunks:", example_rag["retrieved_chunks"])

rag_stage2_df = rag.run_on_dataframe(stage2_questions, k=config.retrieval_default_k)
save_dataframe_csv(rag_stage2_df, config.results_dir / "stage4_rag_stage2_questions.csv")
save_dataframe_json(rag_stage2_df, config.results_dir / "stage4_rag_stage2_questions.json")

comparison_df = build_side_by_side_table(stage2_questions, naive_df, rag_stage2_df)
save_dataframe_csv(comparison_df, config.results_dir / "stage5_comparison.csv")
save_dataframe_json(comparison_df, config.results_dir / "stage5_comparison.json")

comparison_df.head(10)

## 10) Evaluate Correctness with LLM-as-Judge

Use DeepSeek-V3-0324 (configured in environment) to score correct/incorrect with one-sentence rationale.

## 11) Evaluate Faithfulness with Ragas (First 20 Examples)

Use Ragas `faithfulness.score()` with `llm_factory(AsyncOpenAI(...))` on first 20 sorted examples only.

## 12) Compute Retrieval Page Hit@k (k = 1, 3, 5)

Compute whether any retrieved chunk page matches any evidence page for each question.

## 13) Save Artifacts and Export Results

Persist CSV and JSON outputs in `results/` and summarize aggregate metrics.

In [ ]:
rag_full_df = rag.run_on_dataframe(filtered_df, k=config.retrieval_default_k)
save_dataframe_csv(rag_full_df, config.results_dir / "stage6_rag_full.csv")
save_dataframe_json(rag_full_df, config.results_dir / "stage6_rag_full.json")

correctness_df, correctness_accuracy = compute_correctness_judgements(rag_full_df, config)
save_dataframe_csv(correctness_df, config.results_dir / "stage6_correctness.csv")
save_dataframe_json(correctness_df, config.results_dir / "stage6_correctness.json")

faithfulness_df, faithfulness_mean = compute_faithfulness_first_20(rag_full_df, config)
save_dataframe_csv(faithfulness_df, config.results_dir / "stage6_faithfulness.csv")
save_dataframe_json(faithfulness_df, config.results_dir / "stage6_faithfulness.json")

hit_detail_df, hit_summary_df = compute_page_hit_rate(
    questions_df=filtered_df,
    rag_pipeline=rag,
    k_values=[1, 3, 5],
)
save_dataframe_csv(hit_detail_df, config.results_dir / "stage6_hit_detail.csv")
save_dataframe_json(hit_detail_df, config.results_dir / "stage6_hit_detail.json")
save_dataframe_csv(hit_summary_df, config.results_dir / "stage6_hit_summary.csv")
save_dataframe_json(hit_summary_df, config.results_dir / "stage6_hit_summary.json")

metrics_summary = {
    "correctness_accuracy": correctness_accuracy,
    "faithfulness_mean": faithfulness_mean,
    "page_hit_rates": hit_summary_df.to_dict(orient="records"),
}
save_json(metrics_summary, config.results_dir / "stage6_metrics_summary.json")

print("Metrics summary")
print(json.dumps(metrics_summary, indent=2))
hit_summary_df